# BDIS + APEP · Python — rad sa fajlovima i rešenje zadatka

**Big Data infrastruktura i servisi** i **Analiza podataka u elektronskom poslovanju**  
**Prvi čas · praktična vežba i rešenje zadatka**

Notebook ima dve celine:

1. **Rad sa fajlovima:** `pathlib`, tekstualni fajlovi, JSON i CSV.
2. **Rešenje zadatka „Analiza i preporuka proizvoda u maloj online prodavnici“:** svih 10 tačaka iz postavke zadatka.

> **Kako koristiti:** Izvršavajte ćelije redom (`Shift + Enter`). Ako ste menjali promenljive ili dobili `NameError`, u Jupyter-u koristite **Restart Kernel and Run All Cells**. Kod koristi samo Python standardnu biblioteku; nisu potrebni dodatni paketi za `json`, `csv`, `math`, `random` ili `pathlib`.


---
## Deo A — Rad sa fajlovima

### A1. Putanje i direktorijumi (`pathlib`)

Relativne putanje polaze od **trenutnog radnog direktorijuma** (`Path.cwd()`), a ne nužno od mesta gde je fajl notebook-a fizički sačuvan. Operator `/` spaja delove putanje nezavisno od operativnog sistema.

In [ ]:
from pathlib import Path

print("Trenutni radni direktorijum:", Path.cwd())

folder = Path("podaci")
folder.mkdir(parents=True, exist_ok=True)
print("Folder postoji:", folder.is_dir())
print("Apsolutna putanja do foldera:", folder.resolve())

**Zašto `mkdir(parents=True, exist_ok=True)`?**

- `parents=True`: napravi i roditeljske direktorijume ako nedostaju.
- `exist_ok=True`: **ne prijavljuje grešku kada direktorijum već postoji**.
- Ako na istoj putanji postoji *fajl* umesto direktorijuma, greška se i dalje prijavljuje.

Otvaranje fajla sa `"w"` može da napravi novi **fajl**, ali ne i direktorijume iz njegove putanje. Zato prvo obezbeđujemo `podaci/`.

### A2. Upis i čitanje tekstualnog fajla

`with` zatvara fajl po izlasku iz bloka. `"w"` piše i prepisuje postojeći sadržaj, `"r"` čita, a `"a"` dodaje na kraj fajla. Koristimo `encoding="utf-8"` zbog naših slova.

In [ ]:
putanja_txt = folder / "napomena.txt"

with putanja_txt.open("w", encoding="utf-8") as f:
    f.write("Prvi čas: osnove Python-a, JSON i CSV.\n")
    f.write("Primer rada sa datotekama u elektronskom poslovanju.\n")

with putanja_txt.open("r", encoding="utf-8") as f:
    sadrzaj = f.read()

print("Sadržaj fajla:")
print(sadrzaj)

### A3. Podaci u memoriji: lista rečnika

Isti skup porudžbina koristićemo i za **JSON** i za **CSV**. Važno je da nazivi polja budu dosledni: `id`, `kupac`, `kanal`, `iznos`, `placeno`.

In [ ]:
porudzbine = [
    {"id": 101, "kupac": "Ana",   "kanal": "web",    "iznos": 2500.0, "placeno": True},
    {"id": 102, "kupac": "Marko", "kanal": "mobile", "iznos": 1800.0, "placeno": True},
    {"id": 103, "kupac": "Ana",   "kanal": "web",    "iznos": 3200.0, "placeno": False},
]

print("Broj porudžbina:", len(porudzbine))
print("Kupac iz prve porudžbine:", porudzbine[0]["kupac"])
print("Tip podataka:", type(porudzbine))

### A4. JSON — upis i čitanje

JSON čuva strukture kao što su objekti, nizovi, brojevi, stringovi i logičke vrednosti. Modul `json` je sastavni deo Pythona.

- `json.dump(objekat, fajl)` i `json.load(fajl)` rade sa **fajlovima**.
- `json.dumps(objekat)` i `json.loads(tekst)` rade sa **stringovima**.
- `ensure_ascii=False` čuva naša slova čitljivim, a `indent=2` formatira zapis.
- Python `True`/`False` postaju JSON `true`/`false`.

In [ ]:
import json

putanja_json = folder / "porudzbine.json"
putanja_json.parent.mkdir(parents=True, exist_ok=True)

with putanja_json.open("w", encoding="utf-8") as f:
    json.dump(porudzbine, f, ensure_ascii=False, indent=2)

print("Sačuvan JSON:", putanja_json)
print(putanja_json.read_text(encoding="utf-8"))

with putanja_json.open("r", encoding="utf-8") as f:
    ucitane_porudzbine_json = json.load(f)

print("Tip učitanih podataka:", type(ucitane_porudzbine_json))
print("Broj porudžbina:", len(ucitane_porudzbine_json))
print("Kupac iz prve porudžbine:", ucitane_porudzbine_json[0]["kupac"])
print("Tip vrednosti 'iznos':", type(ucitane_porudzbine_json[0]["iznos"]))
print("Tip vrednosti 'placeno':", type(ucitane_porudzbine_json[0]["placeno"]))

### A5. CSV — upis i čitanje

CSV je **tabelarni tekstualni** format. `csv.DictWriter` upisuje rečnike kao redove. Svaki ključ koji se upisuje mora biti naveden u `fieldnames` (osim ako izričito ne zatražimo ignorisanje dodatnih polja). Ovde koristimo **istu šemu** kao u JSON primeru, kako ne bismo dobili `ValueError` zbog nepoznatog polja.

Pri pisanju i čitanju CSV fajla koristimo `newline=""`, prema preporuci Python dokumentacije.

In [ ]:
import csv

putanja_csv = folder / "porudzbine.csv"
putanja_csv.parent.mkdir(parents=True, exist_ok=True)
kolone = ["id", "kupac", "kanal", "iznos", "placeno"]

with putanja_csv.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=kolone)
    writer.writeheader()
    writer.writerows(porudzbine)

print("Sačuvan CSV:", putanja_csv)
print(putanja_csv.read_text(encoding="utf-8"))


with putanja_csv.open("r", newline="", encoding="utf-8") as f:
    ucitane_porudzbine_csv = list(csv.DictReader(f))

print("Prvi učitani red:", ucitane_porudzbine_csv[0])
print("Tip cene posle čitanja CSV-a:", type(ucitane_porudzbine_csv[0]["iznos"]))

iznos_kao_broj = float(ucitane_porudzbine_csv[0]["iznos"])
print("Konvertovan iznos:", iznos_kao_broj)
print("Tip nakon konverzije:", type(iznos_kao_broj))

**Važna razlika:** `json.load()` rekonstruiše tipove kao što su `float` i `bool`, a `csv.DictReader()` uobičajeno vraća **stringove**. Zato numeričku obradu CSV podataka radimo tek posle konverzije, npr. `float(red["iznos"])`.

### A6. Kratka analiza porudžbina

Koliki je zbir iznosa samo **plaćenih** porudžbina?

In [ ]:
ukupno_placeno = 0.0

for porudzbina in porudzbine:
    if porudzbina["placeno"]:
        ukupno_placeno += porudzbina["iznos"]

print(f"Ukupna vrednost plaćenih porudžbina: {ukupno_placeno:.2f} RSD")
# Očekivano: 4300.00 RSD

---
## Deo B — Rešenje zadatka: Analiza i preporuka proizvoda u maloj online prodavnici

**Polazna postavka:** mala online prodavnica želi da predstavi katalog proizvoda, prikaže proizvode dostupne za uneti budžet, pronađe najskuplji proizvod, simulira klik korisnika, izračuna prosek cena i prihod, dopuni katalog i sortira proizvode po ceni.

Rešenje prati **originalnih 10 tačaka zadatka**. Svaka tačka ima posebnu ćeliju. Podaci iz ovog dela namerno su odvojeni od primera porudžbina iz Dela A.

> **Važno:** Ćelije se nadovezuju jedna na drugu. Najpre pokrenite 1. i 2. tačku, pa redom ostale. Za ponovni početak koristite **Restart Kernel and Run All Cells**. U tački 4 biće potrebno da unesete budžet (npr. `100`).

### 1. Kreirati listu proizvoda

Lista sadrži osam naziva proizvoda (`str`).

In [ ]:
proizvodi = [
    "Laptop",
    "Miš",
    "Tastatura",
    "Monitor",
    "Slušalice",
    "Web kamera",
    "USB memorija",
    "Eksterni disk",
]

print("Ukupan broj proizvoda:", len(proizvodi))
print(proizvodi)

### 2. Kreirati rečnik cena proizvoda

Naziv proizvoda je **ključ**, a cena (`float`) njegova **vrednost**.

In [ ]:
cene = {
    "Laptop": 950.99,
    "Miš": 19.99,
    "Tastatura": 45.50,
    "Monitor": 220.00,
    "Slušalice": 65.99,
    "Web kamera": 49.90,
    "USB memorija": 12.50,
    "Eksterni disk": 89.99,
}

print("Cena laptopa:", cene["Laptop"])
print("Tip cene:", type(cene["Laptop"]))

### 3. Ispisati sve proizvode i cene

`enumerate(..., start=1)` numeriše redove od 1. Format `:.2f` prikazuje dve decimale.

In [ ]:
print("KATALOG PROIZVODA")
for i, proizvod in enumerate(proizvodi, start=1):
    print(f"{i}. {proizvod} - {cene[proizvod]:.2f} €")

### 4. Unos budžeta i preporuka dostupnih proizvoda

Funkcija `input()` vraća `str`, pa unos pretvaramo u `float`. **Unesite broj koristeći tačku kao decimalni separator** (npr. `100` ili `99.50`). Proizvod prikazujemo samo ako je cena manja ili jednaka budžetu.

In [ ]:
budzet = float(input("Unesite budžet u evrima (npr. 100): "))

print(f"Proizvodi koje možete da priuštite za {budzet:.2f} €:")
broj_dostupnih = 0

for proizvod in proizvodi:
    if cene[proizvod] <= budzet:
        print(f"- {proizvod}: {cene[proizvod]:.2f} €")
        broj_dostupnih += 1

if broj_dostupnih == 0:
    print("Nema proizvoda koji odgovaraju unetom budžetu.")

### 5. Funkcija `najskuplji_proizvod()`

`max(cene, key=cene.get)` vraća **ključ rečnika** sa najvećom pridruženom vrednošću (cenom), a ne najveći naziv po abecedi.

Postavka zahteva funkciju bez obaveznih argumenata; zato ovde koristi već definisan rečnik `cene`.

In [ ]:
def najskuplji_proizvod():
    return max(cene, key=cene.get)

najskuplji = najskuplji_proizvod()
print(f"Najskuplji proizvod je {najskuplji}, cena: {cene[najskuplji]:.2f} €")

### 6. `random` — simulacija klika korisnika

`random.choice(lista)` bira **jedan slučajan element**. Očekivano je da rezultat nije uvek isti.

In [ ]:
import random

kliknuti_proizvod = random.choice(proizvodi)
print(f"Korisniku je privukao pažnju proizvod: {kliknuti_proizvod}")

### 7. `math` — prosečna cena proizvoda

`math.fsum()` precizno sabira `float` vrednosti. Delimo brojem proizvoda i zatim koristimo **ugrađeni** `round()` za zaokruživanje na dve decimale. `round()` nije deo modula `math`.

In [ ]:
import math

prosecna_cena = math.fsum(cene.values()) / len(cene)
prosecna_cena = round(prosecna_cena, 2)
print(f"Prosečna cena proizvoda: {prosecna_cena:.2f} €")
# Očekivano: 181.86 €

### 8. Broj prodatih komada i ukupan prihod

Druga lista `prodati_komadi` sadrži količine **u istom redosledu** kao lista `proizvodi`. Ukupan prihod je zbir proizvoda `cena × količina`.

In [ ]:
prodati_komadi = [5, 30, 20, 12, 18, 10, 40, 8]

assert len(prodati_komadi) == len(proizvodi), "Svaki proizvod mora imati svoju količinu."

ukupan_prihod = 0.0
for i in range(len(proizvodi)):
    proizvod = proizvodi[i]
    kolicina = prodati_komadi[i]
    ukupan_prihod += cene[proizvod] * kolicina

print(f"Ukupan prihod prodavnice: {ukupan_prihod:.2f} €")
# Očekivano: 11811.39 €

### 9. Dodati novi proizvod

Ažuriramo listu i rečnik. Kako u tački 8 pratimo i prodate komade, za novododati proizvod dopisujemo količinu `0`, da liste ostanu usklađene. Provera `if` sprečava slučajno dupliranje pri ponovnom izvršavanju ove ćelije.

In [ ]:
novi_proizvod = "Tablet"
nova_cena = 350.00

if novi_proizvod not in proizvodi:
    proizvodi.append(novi_proizvod)
    prodati_komadi.append(0)

cene[novi_proizvod] = nova_cena

print("Ažurirani katalog:")
for i, proizvod in enumerate(proizvodi, start=1):
    print(f"{i}. {proizvod} - {cene[proizvod]:.2f} €")

assert len(prodati_komadi) == len(proizvodi)

### 10. Sortirati proizvode po ceni

`sorted()` pravi novu listu. Funkcija `key=lambda proizvod: cene[proizvod]` govori Python-u da poredi proizvode po **ceni**, a ne po nazivu. `lambda` je kratka anonimna funkcija i može se objasniti kao naprednija napomena.

In [ ]:
sortirani_proizvodi = sorted(proizvodi, key=lambda proizvod: cene[proizvod])

print("Proizvodi sortirani od najjeftinijeg ka najskupljem:")
for i, proizvod in enumerate(sortirani_proizvodi, start=1):
    print(f"{i}. {proizvod} - {cene[proizvod]:.2f} €")

---
## Provera rešenja i ponavljanje

Ako ste prošli svih 10 koraka redom, očekivani rezultati su:

- **Najskuplji proizvod:** Laptop — **950.99 €**.
- **Prosečna cena pre dodavanja Tablet-a:** **181.86 €**.
- **Ukupan prihod pre dodavanja Tablet-a:** **11811.39 €**.
- **Broj proizvoda nakon dodavanja Tablet-a:** **9**.
- **Sortiranje:** prvi je USB memorija (12.50 €), poslednji Laptop (950.99 €).

`random.choice()` nema fiksan rezultat. Rezultat u tački 4 zavisi od budžeta koji korisnik unese.

**Za razgovor sa studentima:**

1. Zašto u tački 4 `input()` pretvaramo u `float`?
2. Zašto u CSV-u cena stiže kao `str`, a nakon `json.load()` kao `float`?
3. Koja je razlika između liste `proizvodi` i rečnika `cene`?
4. Šta će se promeniti ako ponovo pokrenemo ćeliju za upis JSON fajla?
5. Zašto je korisno izvršiti ceo notebook od početka pre predaje?

**Napomena za nastavu:** Ovo je rešenje originalne postavke (tačke 1–10). Rad sa fajlovima je u **odvojenom delu A**, da se originalni zahtevi zadatka ne menjaju.